# 📚 LangChain ile Retrieval Augmented Generation'a Giriş 🦜🔗

Bu not defterinde LangChain kullanarak Retrieval Augmented Generation'ı nasıl kullanacağınızı öğreneceksiniz.

Kendi belgelerimiz hakkında sorular sormak için bir LLM kullanacağız!

## ⚙️ Kurulum

👉 Temel kütüphaneleri içe aktarmak için aşağıdaki hücreyi çalıştırın.

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pprint import pprint
from IPython.display import Markdown

👉 API anahtarımızı tekrar yüklemek için aşağıdaki hücreyi çalıştırın:

In [2]:
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file

True

## 📚 Neden RAG?

Bir LLM kendi başına, öğrendiği her şey hakkında sorulara yanıt verebilir.

Bunun birkaç dezavantajı vardır:
- Eğitim verileri geçmişten gelir ve en son verilerle güncellenmez.
- Sadece eğitim aldığı verileri bilir.

Bir LLM'yi kendi verilerimizle çalışması için kullanmak istiyoruz. İşte bu noktada RAG (Retrieval-Augmented Generation) devreye girer.

1. **Retrieval-Augmented Generation (RAG)**, gerçek doğruluğu artırmak için bir dil modelini belge alıcı ile birleştirir.
2. **İlgili dış belgeleri alır** (örneğin, bilgi tabanından) yanıtlar üretmeden önce.
3. **Dil modeli hem istemi hem de alınan bağlamı kullanarak** daha bilgili ve temelli çıktılar üretir.

## 🇪🇺 Bağlam

Bu meydan okumada, Avrupa Parlamentosu'ndan belgelerle çalışacağız.

Bir gazeteci olduğunuzu ve Avrupa Parlamentosu'nun genel kurul oturumları sırasında belirli bir konu hakkında neler söylendiğini öğrenmek istediğinizi düşünün. Bu oturumlar yılda 12 kez Strasbourg'da gerçekleşir ve 4 gün sürer. Oturumların transkriptleri EP'nin web sitesinde mevcuttur.

Kesinlikle tüm bu transkriptleri karıştırmak istemezsiniz. O halde, hayatımızı kolaylaştırmak için RAG'ı kullanalım!

Bu, her zaman test etmek için yepyeni veriler alabileceğimiz için çalışmak üzere iyi verilerdir.

## 📘 Verileri alalım

1. [EP'nin web sitesine](https://www.europarl.europa.eu/plenary/en/debates-video.html) gidin. 
1. Bu sizi en son genel kurul oturumuna yönlendirecektir.
1. İlk tarihin altında, "▶️ Verbatim reports HTML" bölümünde `HTML`'e tıklayın.
1. Sayfanın sonuna kaydırın ve alttaki PDF dosyasını indirin.
1. Dosyayı `data` klasörüne kaydedin.

Bir belgeyle başlayacağız, ancak daha sonrası için diğer birkaç günün aynısını şimdiden indirebilirsiniz.

Belgeye bir göz atın. Kaç sayfası var? Belge hakkında bir fikir edinmek için hızlıca belgede gezinin.

## 🔢 Belgeleri gömme

Belgeleri gömmek, tüm belgeleri veya belge parçalarını vektörlere çevirmek anlamına gelir.

LangChain🦜🔗 yine çok yardımcı olacak.

Bir gömme aracı (embedder) başlatalım ve deneyelim. LLM olarak Gemini kullandığımız için, Google'ın metin gömme araçlarında kalalım.

In [3]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

👉 Basit bir metin parçasını gömmek için gömme aracının `.embed_query()` metodunu deneyin.

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

👉 Bu `sample_embedding`'i keşfetmek için zaman ayırın. Nasıl görünüyor? Tipi nedir? Gömme boyutu nedir?

In [5]:

sample_embedding = embeddings.embed_query(
    "What is the capital of France?"
)

print(sample_embedding[:10])


[-0.032554302364587784, 0.013054960407316685, 0.015067406930029392, -0.07128717750310898, -0.03106236644089222, -0.019940834492444992, -0.013077609241008759, -0.009117965586483479, 0.03893592208623886, -0.019308630377054214]


## 💾 PDF'den gerçek verilerimizi yükle

Artık bir gömmenin nasıl göründüğünü biliyoruz, gerçek verilerimizle çalışmanın zamanı geldi.

👉 [LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/document_loaders/index#pdfs) gidin ve PyPDF kullanarak bir PDF'yi nasıl yükleyebileceğinizi öğrenin.

👉 Sonra devam edin ve daha önce indirdiğiniz PDF'lerden birini yükleyin.

In [7]:

from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

pdf_files = list(Path("data").glob("*.pdf"))

if not pdf_files:
    raise FileNotFoundError("data klasöründe PDF bulunamadı!")

pdf_path = pdf_files[0]

loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

print("PDF:", pdf_path.name)
print("Sayfa sayısı:", len(pages))


PDF: CRE-10-2026-11-23_EN.pdf
Sayfa sayısı: 102


👉 `pages`'i keşfedin:
- Veri tipi nedir?
- Kaç sayfanız var?
- Bir sayfanın tipi nedir?
- Bir sayfanın içeriğine nasıl erişebilirsiniz?
- Tam belgenin kaç karakteri var?
- Bir sayfanın `metadata`'sında neler var?

In [10]:

print("Veri tipi:", type(pages))
print("Sayfa sayısı:", len(pages))
print("İlk sayfanın tipi:", type(pages[0]))
print("İlk 500 karakter:", pages[0].page_content[:500])
print("Toplam karakter:", sum(len(p.page_content) for p in pages))
print("Metadata:", pages[0].metadata)


Veri tipi: <class 'list'>
Sayfa sayısı: 102
İlk sayfanın tipi: <class 'langchain_core.documents.base.Document'>
İlk 500 karakter: 2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN
Toplam karakter: 293124
Metadata: {'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '724191', 'dmxml.render.traceid': '6b05c426656f03d2cc09bb7343af8865', 'uid': 'eu.europa.europarl-DIN1-2026-0000316703_01.00-xm-01.00_text-xml', 'source': 'data\\CRE-10-2026-11-23_

## ✂️ Verilerimizi böl

Tam belgemiz gömülmek için çok uzun. Metin gömme aracımız 2.048 tokena kadar giriş alabilir. Gemini modelleri için bu yaklaşık 8.196 karakterdir (token başına 4 karakter).

Bu yüzden belgemizi daha küçük parçalara bölmek istiyoruz.

Zaten çalışabileceğimiz bir dizi sayfamız var. Ama sayfa sonları biraz keyfi: genellikle cümlenin ortasında görünürler.

Ayrıca, sayfalar arasında örtüşme yoktur. Bu yüzden bir sayfanın ilk satırı önceki tüm bağlamı kaçırır. Tam metni biraz örtüşmeyle bölmek daha iyidir.

İlk olarak, PDF'yi tekrar yükleyeceğiz, bu sefer bölmeden.

In [11]:

loader = PyPDFLoader(str(pdf_path), mode="single")

pdf = loader.load()
pdf_text = pdf[0].page_content

print("Toplam karakter:", len(pdf_text))


Toplam karakter: 293326


Artık tüm PDF'imizi tek bir belge olarak aldığımıza göre, onu daha akıllı bir şekilde parçalara bölebiliriz.

👉 Yine, ["Özyinelemeli olarak bölme" konusundaki LangChain belgelerine](https://docs.langchain.com/oss/python/integrations/splitters/recursive_text_splitter) gidin ve `pdf` _belgelerimizi_ parçalara (LangChain'de `documents` olarak adlandırılır) nasıl böleceğinizi öğrenin.

2_000 karakter (bizim durumumuzda yaklaşık yarım sayfa) parçalara 400 örtüşmeyle bölün. İsterseniz diğer değerlerle deneyebilirsiniz.

`RecursiveCharacterTextSplitter`'ın `.split_documents()` metodunu kullanın: bu metod giriş olarak bir belge alır ve bölünmüş belgeler çıktısı verir.

In [12]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)

all_splits = text_splitter.split_documents(pdf)

print("Toplam parça:", len(all_splits))
print("İlk parça:", all_splits[0].page_content[:300])


Toplam parça: 184
İlk parça: 2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG V


👉 `all_splits`'i inceleyin:
- Veri tipi nedir?
- Kaç bölümünüz var?
- Bir bölümün tipi nedir?
- Bir bölümün içeriğine nasıl erişebilirsiniz?
- Şimdi toplamda kaç karakterimiz var?
- Bir bölümün `metadata`'sında neler var?

In [13]:

print("Veri tipi:", type(all_splits))
print("Toplam parça sayısı:", len(all_splits))
print("Bir parçanın tipi:", type(all_splits[0]))

print("\nİlk parçanın içeriği:")
print(all_splits[0].page_content[:500])

print("\nToplam karakter sayısı:")
print(sum(len(doc.page_content) for doc in all_splits))

print("\nİlk parçanın metadata bilgileri:")
print(all_splits[0].metadata)


Veri tipi: <class 'list'>
Toplam parça sayısı: 184
Bir parçanın tipi: <class 'langchain_core.documents.base.Document'>

İlk parçanın içeriği:
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN

Toplam karakter sayısı:
358238

İlk parçanın metadata bilgileri:
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '724191', 'dmxml.render.traceid': '6b05c426656f03d2cc09bb7343af8865', 'uid': 'eu.europa.europarl-DIN1-2026-0000316703_01.00-xm-01.00_te

## 🗄️ Her şeyi bir araya getir: belgelerimizi gömme ve vektör deposunda sakla

Elimizde şunlar var:
- Bir gömme aracı
- Veriyi yüklemek için bir yükleyici
- Belgemizi belgelere bölmek için bir metin bölücü

Neyi kaçırıyoruz?

Belgelerimizi gömebiliriz, ama onları bir yerde saklamak istiyoruz. İşte burada vektör deposu devreye girer: şunları saklamamıza olanak sağlar:
- belgeyi (parçayı),
- onun gömmesini,
- meta verilerini.

Sonraki adımda belgeleri verimli bir şekilde alabilecek olacağız.

👉 Bir `InMemoryVectorStore` nasıl oluşturabileceğinizi görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

In [14]:

from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embedding=embeddings)

document_ids = vector_store.add_documents(
    documents=all_splits
)

print("Eklenen belge sayısı:", len(document_ids))


Eklenen belge sayısı: 184


In [15]:

print(document_ids[:3])


['c986d604-49e1-46d5-8187-10ea0a38db08', 'a9c23ed5-ad48-4fbf-b545-8e9e22731a36', 'c3436e91-de48-46a7-a4e9-eafa75cd2b83']


In [16]:

retrieved_by_ids = vector_store.get_by_ids(
    document_ids[:3]
)

print(retrieved_by_ids)


[Document(id='c986d604-49e1-46d5-8187-10ea0a38db08', metadata={'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '724191', 'dmxml.render.traceid': '6b05c426656f03d2cc09bb7343af8865', 'uid': 'eu.europa.europarl-DIN1-2026-0000316703_01.00-xm-01.00_text-xml', 'source': 'data\\CRE-10-2026-11-23_EN.pdf', 'total_pages': 102}, page_content='2024-2029 \n \n \nПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA \nACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA \nDOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE \nFULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI \nAUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN \nISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD \nΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES \nVERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR \nCOMPTE RENDU IN EXTENSO DES DÉBATS  DOSLOVNÝ ZÁPIS Z ROZPRÁV \n

👉 Bir vektör deposundaki belgenin içeriğine ve meta verilerine nasıl erişebilirsiniz?

In [17]:

print("Belge içeriği:")
print(retrieved_by_ids[0].page_content[:500])

print("\nMetadata bilgileri:")
print(retrieved_by_ids[0].metadata)


Belge içeriği:
2024-2029 
 
 
ПЪЛЕН ПРОТОКОЛ НА РАЗИСКВАНИЯТА  DEBAŠU STENOGRAMMA 
ACTA LITERAL DE LOS DEBATES  POSĖDŽIO STENOGRAMA 
DOSLOVNÝ ZÁZNAM ZE ZASEDÁNÍ  AZ ÜLÉSEK SZÓ SZERINTI JEGYZŐKÖNYVE 
FULDSTÆNDIGT FORHANDLINGSREFERAT  RAPPORTI VERBATIM TAD-DIBATTITI 
AUSFÜHRLICHE SITZUNGSBERICHTE  VOLLEDIG VERSLAG VAN DE VERGADERINGEN 
ISTUNGI STENOGRAMM  PEŁNE SPRAWOZDANIE Z OBRAD 
ΠΛΗΡΗ ΠΡΑΚΤΙΚΑ ΤΩΝ ΣΥΖΗΤΗΣΕΩΝ  RELATO INTEGRAL DOS DEBATES 
VERBATIM REPORT OF PROCEEDINGS STENOGRAMA DEZBATERILOR 
COMPTE RENDU IN

Metadata bilgileri:
{'producer': 'Aspose.Words for Java 24.2.0', 'creator': 'Aspose.Words', 'creationdate': '', 'author': 'e-Parliament@europarl.europa.eu', 'dmxml.render.id': '724191', 'dmxml.render.traceid': '6b05c426656f03d2cc09bb7343af8865', 'uid': 'eu.europa.europarl-DIN1-2026-0000316703_01.00-xm-01.00_text-xml', 'source': 'data\\CRE-10-2026-11-23_EN.pdf', 'total_pages': 102}


## 🔎 Benzer belgeleri almak için vektör deposunu kullan

Artık belgeleri gömleğe çevirdiğimize göre, benzer belgeleri almak için vektör deposunu kullanabiliriz.

👉 Bunun nasıl çalıştığını görmek için ["Vektör depoları" üzerine LangChain belgelerini](https://docs.langchain.com/oss/python/langchain/knowledge-base#3-vector-stores) kontrol edin.

Bir sorgu kullanın, örneğin "Tarım politikası üzerine tartışmayı özetle.", ve en benzer belgeleri bulun. Ayrıca alınacak belge sayısını da belirtebilirsiniz.

In [18]:

query = "Summarize the discussions about European democracy."

retrieved_docs = vector_store.similarity_search(
    query,
    k=4
)

print("Bulunan belge sayısı:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\nBelge {i}:")
    print(doc.page_content[:400])


Bulunan belge sayısı: 4

Belge 1:
rinkos monopolizavimą. Todėl būtinai reikia peržvalgos, kuri iš tikrųjų reglamentuotų aiškiai, 
labai aiškiai, Paslaugų direktyvos kontekste ir įvestų antimonopolines priemones tiems, kurie 
uzurpuoja žiniasklaidą.
46  23-11-2026 
1-0070-0000 
Arkadiusz Mularczyk (ECR). – Pani Przewodnicząca! Nie można bronić demokracji, stosując 
podwójne standardy w debacie publicznej. Przypomnę przykład Polski

Belge 2:
important to stress that it is both. 
 
That is essentially the Triple Lock. So the Triple Lock was built in to the Irish people's acceptance 
of Nice the following year, in a referendum, by plebiscite of the people. And now the government 
are saying that they can just forget about that – including the current Taoiseach, who was in that 
government, who is saying: 'That doesn't matter, expediency

Belge 3:
démocratie confie à des gens que personne n'a élus le pouvoir de dire ce qu'un peuple a le droit 
d'entendre? Les Européens n'ont pas besoin qu'

Bu, RAG'ın sözde "Alma" (Retrieval) kısmını tamamlar: artık sorgumuza en benzer belgeleri bulabiliriz.

Çalışmanın çoğu artık tamamlandı!

## 💬 Sorumuza bir cevap üret

Şimdiye kadar benzer belgeleri almamızı sağlamak için sadece bir **gömme modeli** kullandık.

Şimdi, sorumuzla bir cevap almak için üretici bir LLM kullanacağız: ona aldığımız belgeler ve sorumuzla besleyeceğiz.

Bunu yapmanın en temel yolu tüm girdilerimizi birbirine bağlamak, sorumuzla eklemek ve sonucu görmek olacaktır.

Bir deneyelim.

👉 İlk olarak önceki meydan okumalarda olduğu gibi bir LLM başlatın.

In [19]:

from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "gemini-3.1-flash-lite",
    model_provider="google_genai",
    vertexai=True,
    project="gen-lang-client-0260388601",
    location="global"
)

print("LLM hazır!")


LLM hazır!


Sonra temel bir istem oluşturun:

In [20]:
prompt = '\n\n'.join([doc.page_content for doc in retrieved_docs])
prompt += "\n\n" + query

👉 Şimdi istemi kullanın:

In [21]:

response = llm.invoke(prompt)

display(Markdown(response.text))


The provided excerpts from the European Parliament session on November 23, 2026, reflect a deeply polarized debate regarding the state of democracy within the European Union and its member states. The discussions centered on three primary themes:

**1. Concerns over Media Control and Regulation**
*   **Market Concentration:** There was a call for clearer, more stringent regulations under the Audiovisual Media Services Directive (AVMSD) to prevent the monopolization of media and the "usurpation" of information channels.
*   **EU Oversight:** Executive Vice-President Henna Virkkunen acknowledged the need to review the AVMSD to ensure a "level playing field." She highlighted the necessity of regulating new digital actors—specifically social media influencers—and enhancing the protection of minors from harmful online content (cyberbullying, self-harm, etc.).
*   **Accusations of Bias:** Some members argued that EU efforts to protect democracy are, in reality, mechanisms of manipulation or censorship. Critics suggested that terms like "European democracy shield" are used to suppress alternative viewpoints, silence conservative voices, and push a specific ideological agenda.

**2. Accusations of Political Interference and Double Standards**
*   **National Sovereignty:** Polish MEP Arkadiusz Mularczyk criticized the European Commission for remaining silent on the current liberal government’s actions in Poland (which he characterized as "brutal" and an illegal takeover of public institutions), while simultaneously noting that foreign actors had heavily influenced previous elections in Poland. He argued that the EU applies "double standards" to undermine conservative governments.
*   **Democratic Integrity:** Other speakers emphasized that democracy is not merely about winning elections but about the protection of the defeated, the independence of the judiciary, and the separation of powers. They cautioned that the path from winning an election to undermining democratic institutions can be alarmingly short.

**3. Immigration and Social Polarization**
*   **"Remigration" and Censorship:** MEP Isabella Tovaglieri (PfE) claimed that the Parliament suppressed a debate on "remigration." She criticized the EU for focusing on trivial regulations (like bottle caps) while ignoring the "devastating effects of mass immigration," arguing that the current trajectory is harmful to the safety of European cities.
*   **External Challenges:** There were critiques of how the EU manages its global standing, with some members suggesting that the bloc is trapped in a cycle of competition with the US, Russia, and China, and that democratic rhetoric is often a mask for broader geopolitical or economic interests.

**Summary Conclusion:**
The debate revealed a fundamental divide between those who believe the EU must proactively regulate media and digital spaces to protect democratic resilience and those who view these regulations as a form of state-sponsored censorship and ideological control. There is a palpable tension between the desire for centralized European standards and the assertion of national sovereignty, with mutual accusations of democratic backsliding and bias shaping the discourse.

Bu fena değil, ama modele daha fazla rehberlik vererek daha kapsamlı bir istem yazarak daha iyisini yapabiliriz.

Bunu yapan ilk kişiler biz değilmişiz ve LangChain'in bizim için önceden hazırlanmış istem kütüphanesi var.

👉 Aşağıdaki hücreyi çalıştırın ve nasıl çalıştığını anlamaya çalışın. (LangSmithMissingAPIKeyWarning hakkında bir uyarı alacaksınız, bunu görmezden gelebilirsiniz.)

In [22]:
from langchain_classic import hub

prompt_template = hub.pull("rlm/rag-prompt")

example_messages = prompt_template.invoke(
    {"context": "(context goes here)", "question": "(question goes here)"}
).to_messages()

print("\n")
print(example_messages[0].content)



You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: (question goes here) 
Context: (context goes here) 
Answer:


LangChain'in bizim için nasıl daha kesin bir istem oluşturduğunu görüyor musunuz? Bunu RAG'ımız için kullanalım!

👉 İlk olarak, tüm alınan belgeleri iki yeni satırla ayrılmış tek bir uzun dizgiye birleştirin.

In [23]:

docs_content = "\n\n".join(
    doc.page_content for doc in retrieved_docs
)

print("Toplam karakter:", len(docs_content))


Toplam karakter: 7828


👉 Sonra, sorgunuz ve alınan belgelerden başlayarak bir `prompt` oluşturun. Yukarıdaki örneğe bakmayı unutmayın.

In [24]:

the_prompt = prompt_template.invoke({
    "context": docs_content,
    "question": query
})

print(the_prompt)


messages=[HumanMessage(content="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: Summarize the discussions about European democracy. \nContext: rinkos monopolizavimą. Todėl būtinai reikia peržvalgos, kuri iš tikrųjų reglamentuotų aiškiai, \nlabai aiškiai, Paslaugų direktyvos kontekste ir įvestų antimonopolines priemones tiems, kurie \nuzurpuoja žiniasklaidą.\n\x0c46  23-11-2026 \n1-0070-0000 \nArkadiusz Mularczyk (ECR). – Pani Przewodnicząca! Nie można bronić demokracji, stosując \npodwójne standardy w debacie publicznej. Przypomnę przykład Polski w 2023 roku, kiedy \nrządził konserwatywny rząd, bardzo wielu aktorów zagranicznych i USAid, i Komisja Europejska \ni wiele, wiele innych funduszy zaangażowanych było w wybory w Polsce. Przeliczono, że były to \nsetki milionów dolarów, ale też eu

👉 Son olarak az önce oluşturduğumuz `the_prompt` ile LLM modelini kullanın:

In [25]:

response = llm.invoke(the_prompt)

display(Markdown(response.text))


Discussions on European democracy focus on the critical need to protect media pluralism, maintain the independence of institutions, and ensure a fair public debate. Speakers expressed concerns regarding external interference, the alleged censorship of dissenting political views, and the erosion of democratic standards by governments. Additionally, there is an emphasis on regulating digital spaces and media services to foster resilience against misinformation and protect citizens, particularly minors.

🎉 İlk RAG'ımızı tamamladık: LLM kendisine sağladığımız belgelerde ***temelli*** metin üretti.

## 💾 Gömmelerimizi kalıcı hale getir

Şimdiye kadar bellekte vektör deposuyla çalıştık. Bu yüzden not defterinizi kapattığınızda, tüm gömmeleri de kaybedeceksiniz.

⚠️ Bu gömmelerin sağlayıcınızın platformunda, bu durumda Google'ın makinelerinde çalışan modeller tarafından üretildiğini unutmayın. Ve bedava çalışmazlar. 💰

Bunun gibi bir, nispeten küçük belge için maliyet düşüktür, ama hızla artar. Şimdiye kadar sadece bir günün transkriptleriyle çalıştık. Oturum başına 3 tane daha, yılda 12 oturum, birden fazla yıl var...

Bunu çözmek için sadece vektör depomuzla kalıcı bir taneyi değiştireceğiz. Bu LangChain'in avantajıdır: bileşenleri değiştirmek çok kolay.

Bellekteki vektör depomuz deneme için harikaydı, şimdi başka bir taneyle değiştireceğiz. Çok popüler bir vektör deposu olan [Chroma](https://www.trychroma.com/)'yı kullanacağız. Bunu yerel olarak çalıştırabilir ve LangChain aracılığıyla kullanabiliriz.

Tüm akışımızı yeniden oluşturacağız. Her şeyi birkaç kod hücresinde tekrar bir araya getirmeye çalışmak iyi bir alıştırmadır. Aynı zamanda her şeyi yeniden kullanılabilir koda dönüştüreceğiz.

Sonunda iki fonksiyon istiyoruz:

1. `embed_and_store()`: Başka bir oturumun transkriptini vektör veritabanımıza ekle, böylece alacağımız daha fazla veri olsun.
2. `answer()`: Vektör depomuzla farklı sorularla sorgula.

#### 1. Bir Chroma vektör deposu başlat

👉 **Veri kalıcılığıyla** (yani verileri diskteki bir dizinde saklayarak) Chroma vektör deposunun nasıl oluşturulacağını görmek için [LangChain'in belgelerine](https://python.langchain.com/docs/integrations/vectorstores/chroma/) bakın.

In [26]:

from langchain_chroma import Chroma

chroma_store = Chroma(
    collection_name="european_parliament_rag",
    embedding_function=embeddings,
    persist_directory="./chroma_db"
)

print("Chroma veritabanı hazır!")


Chroma veritabanı hazır!


#### 2. `embed_and_store()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [27]:

def embed_and_store(file_path, vector_store):
    """Load a PDF, split it and store the chunks."""

    loader = PyPDFLoader(str(file_path), mode="single")
    pdf = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=400
    )

    all_splits = splitter.split_documents(pdf)

    document_ids = vector_store.add_documents(all_splits)

    return document_ids


👉 Fonksiyonunuzu bir dosya veya hatta iki dosyayla deneyin:

In [28]:

stored_ids = embed_and_store(pdf_path, chroma_store)

print("Kaydedilen parça sayısı:", len(stored_ids))
print("İlk 3 ID:", stored_ids[:3])


Kaydedilen parça sayısı: 184
İlk 3 ID: ['c7c08b66-45c1-4a71-9788-a52075a39259', '599ee464-9081-480e-9273-284fcdc46d7d', 'f4f618b5-2b42-4784-8caf-7163aee2fe13']


#### 3. `answer()` oluştur

👉 Bu fonksiyon için kodu tamamlayın:

In [30]:
def answer(query, vector_store, llm, prompt_template=None):
    """Answer a query using the vector store and the language model."""
    # Retrieve similar documents from the vector store
    retrieved_docs = vector_store.similarity_search(query, k=6)

    # Create the prompt
    docs_content = "\n\n".join(doc.page_content for doc in retrieved_docs)

    # If no prompt template is provided, use the default one
    if not prompt_template:
        prompt_template = hub.pull("rlm/rag-prompt")

    prompt = prompt_template.invoke(
        {"context": docs_content, "question": query}
    )

    # Get the answer from the language model
    answer = llm.invoke(prompt)

    return answer.content

👉 Fonksiyonunuzu beğendiğiniz bir sorguyla deneyin:

In [31]:

question = "Summarize the main debates in the European Parliament."

result = answer(
    query=question,
    vector_store=chroma_store,
    llm=llm,
    prompt_template=prompt_template
)

print(result)


[{'type': 'text', 'text': "The main debates in the European Parliament included the revision of the Audiovisual Media Services Directive to strengthen democracy and the protection of EU financial interests through a revised anti-fraud architecture. Additionally, members debated the 35th anniversary of the destruction of Vukovar and the International Court of Justice's judgment regarding Myanmar. Other topics addressed in speeches included concerns over deportation centers in Rwanda and institutional standards.", 'extras': {'signature': 'AY89a19pQqQmwqRjUGcUdN4ZqvNQxPPk8cEIgKKrpjuBSPakxFk8sZTkv2V1KWZS2g8pIBrgTK7rSZ1LjvHu0ghjbGbWuX2LTXsCtTbgtMADxsZUlLWC+lCUG6VU9ss4ZK23UZLslEQBGIgeQ0QjmlL7/ObWf5FfnW1/+5TD6kM/bFGr'}}]


🏁 Tebrikler! Artık LangChain kullanarak RAG'da ustalaştınız ve vektör deponuza daha fazla belge eklemek ve onu sorgulamak için yeniden kullanılabilir fonksiyonlar yapmayı öğrendiniz.

## [İsteğe Bağlı] Meta veri ekleme

Kurduğumuz RAG, vektör deposundaki tüm belgeleri sorgular. Orada birden fazla yılın bilgisinin olduğunu düşünün. Yıllara veya tarihlere göre filtreyebilsek kullanışlı olurdu, değil mi?

Bunu nasıl yaparız? Vektör deposundaki belgelerin meta veri içerdiğini unutmayın. Eğer tarihi ekleyebilseydik, daha sonra filtrelemek için kullanabilirdik.

İpucu: Meta verilerinizi pipeline'ınızda olabildiğince erken ekleyin. Verileriniz vektör deposunda saklandıktan sonra eklemeye çalışmayın.

👉 `embed_and_store()` fonksiyonunuzu uyarlayın.

In [32]:

def embed_and_store_fancy(file_path, vector_store, session_date):
    """Load, split and store a PDF with session date metadata."""

    loader = PyPDFLoader(str(file_path), mode="single")
    documents = loader.load()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=2000,
        chunk_overlap=400
    )

    splits = splitter.split_documents(documents)

    for split in splits:
        split.metadata["session_date"] = session_date

    document_ids = vector_store.add_documents(splits)

    return document_ids


👉 Fonksiyonunuzu deneyin ve vektör deponuzun ek meta veri içerdiğini kontrol edin.

In [33]:

fancy_ids = embed_and_store_fancy(
    pdf_path,
    chroma_store,
    session_date="2026-11-23"
)

print("Kaydedilen belge sayısı:", len(fancy_ids))

sample_doc = chroma_store.get_by_ids(fancy_ids[:1])[0]
print("Metadata:", sample_doc.metadata)


Kaydedilen belge sayısı: 184
Metadata: {'session_date': '2026-11-23', 'producer': 'Aspose.Words for Java 24.2.0', 'dmxml.render.traceid': '6b05c426656f03d2cc09bb7343af8865', 'total_pages': 102, 'creationdate': '', 'creator': 'Aspose.Words', 'dmxml.render.id': '724191', 'uid': 'eu.europa.europarl-DIN1-2026-0000316703_01.00-xm-01.00_text-xml', 'author': 'e-Parliament@europarl.europa.eu', 'source': 'data\\CRE-10-2026-11-23_EN.pdf'}


Şimdi alıcıyı kullanıcının sorduğu tarihe göre sınırlamamız gerekiyor.

👉 `answer()` fonksiyonunuzu bir tarih alabilecek ve yeni meta verilere dayalı olarak belgeleri filtreleyebilecek şekilde uyarlayın.

In [34]:

def answer_by_date(query, vector_store, llm, session_date, prompt_template):
    """Answer a question using documents from a specific date."""

    retrieved_docs = vector_store.similarity_search(
        query,
        k=6,
        filter={"session_date": session_date}
    )

    if not retrieved_docs:
        return "Bu tarihe ait belge bulunamadı."

    docs_content = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = prompt_template.invoke({
        "context": docs_content,
        "question": query
    })

    response = llm.invoke(prompt)

    return response.text


In [35]:

result = answer_by_date(
    query="What topics were discussed in the European Parliament?",
    vector_store=chroma_store,
    llm=llm,
    session_date="2026-11-23",
    prompt_template=prompt_template
)

display(Markdown(result))


The European Parliament discussed several topics, including the revision of the Audiovisual Media Services Directive to strengthen democracy and the protection of the Union's financial interests post-2027. Members also debated the 35th anniversary of the fall of Vukovar and the International Court of Justice’s judgment regarding Myanmar. Additionally, proceedings covered legislative acts concerning medicinal products, passenger rights, and economic governance frameworks.

Harika! Güçlü bir RAG sistemi oluşturmak için benzerlik aramasını meta veri aramasıyla birleştirdiniz!